# Experiment 12 — Rao local 3×3×3 radiomics features + XGBoost

Purpose: compare the **classifier** while keeping Rao's local feature representation.

Input is the output of Experiment 11:
- 577 BraTS 2021 patients
- FLAIR only
- MGMT target at patient level
- local 3×3×3 PyRadiomics
- 93 `%A` activation features per patient

The primary XGBoost configuration is the thesis configuration:
`n_estimators=500, max_depth=6, learning_rate=0.05, min_child_weight=1,
gamma=0, subsample=0.8, colsample_bytree=0.8, reg_alpha=0, reg_lambda=1`.

For a clean comparison, the same 10 stratified folds are reused for all conditions.


## 1. Install

In [ ]:
# !pip install numpy pandas scikit-learn xgboost


## 2. Imports

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, balanced_accuracy_score,
    matthews_corrcoef, confusion_matrix
)
from sklearn.utils.class_weight import compute_class_weight
from xgboost import XGBClassifier


## 3. Configuration

In [ ]:
RANDOM_STATE = 42
N_SPLITS = 10

# Output of Experiment 11 / Rao reproduction notebook
LOCAL_FEATURES_CSV = Path("outputs_rao2021/local_93_percent_activation.csv")

OUTDIR = Path("outputs_exp12_rao_features_xgboost")
OUTDIR.mkdir(parents=True, exist_ok=True)

# Reuse one deterministic set of folds.
FOLD_ASSIGNMENT_CSV = OUTDIR / "fold_assignment_10fold.csv"

# Run both conditions:
# - None = isolates model replacement more cleanly
# - Balanced = mirrors the thesis weighting approach
WEIGHT_CONDITIONS = ["None", "Balanced"]


## 4. Load Rao local features

In [ ]:
df = pd.read_csv(LOCAL_FEATURES_CSV)

required = {"patient_id", "mgmt"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing columns: {missing}")

df["patient_id"] = df["patient_id"].astype(str)
df["mgmt"] = df["mgmt"].astype(int)

feature_cols = [c for c in df.columns if c not in {"patient_id", "mgmt"}]
feature_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(df[c])]

# Keep complete rows for a fixed 93-feature matrix.
work = df.dropna(subset=feature_cols + ["mgmt"]).reset_index(drop=True)

print("Patients:", len(work))
print("Features:", len(feature_cols))
print(work["mgmt"].value_counts().sort_index())
display(work.head())


## 5. Create/reuse identical 10-fold assignments

In [ ]:
def create_fold_assignment(work, path):
    if path.exists():
        folds = pd.read_csv(path)
        return folds

    skf = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    rows = []
    y = work["mgmt"].to_numpy()

    for fold, (_, va_idx) in enumerate(skf.split(np.zeros(len(y)), y), start=1):
        for idx in va_idx:
            rows.append({
                "patient_id": work.iloc[idx]["patient_id"],
                "fold": fold,
            })

    folds = pd.DataFrame(rows).sort_values(["fold", "patient_id"])
    folds.to_csv(path, index=False)
    return folds

folds = create_fold_assignment(work, FOLD_ASSIGNMENT_CSV)
work = work.merge(folds, on="patient_id", how="inner", validate="one_to_one")

display(work[["patient_id", "mgmt", "fold"]].head())
print(work.groupby(["fold", "mgmt"]).size().unstack(fill_value=0))


## 6. Thesis XGBoost configuration

In [ ]:
def build_xgb():
    return XGBClassifier(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        min_child_weight=1,
        gamma=0,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0,
        reg_lambda=1,
        objective="binary:logistic",
        eval_metric="aucpr",
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )


## 7. Run 10-fold CV

In [ ]:
def specificity(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    return tn / (tn + fp) if (tn + fp) else np.nan

rows = []

for weight_condition in WEIGHT_CONDITIONS:
    for fold in range(1, N_SPLITS + 1):
        train = work[work["fold"] != fold]
        valid = work[work["fold"] == fold]

        Xtr = train[feature_cols].to_numpy(dtype=np.float32)
        ytr = train["mgmt"].to_numpy(dtype=int)
        Xva = valid[feature_cols].to_numpy(dtype=np.float32)
        yva = valid["mgmt"].to_numpy(dtype=int)

        sample_weight = None
        if weight_condition == "Balanced":
            classes = np.array([0,1])
            cw_vals = compute_class_weight("balanced", classes=classes, y=ytr)
            cw = {0: float(cw_vals[0]), 1: float(cw_vals[1])}
            sample_weight = np.array([cw[int(v)] for v in ytr], dtype=np.float32)

        model = build_xgb()
        model.fit(Xtr, ytr, sample_weight=sample_weight)

        prob = model.predict_proba(Xva)[:, 1]
        pred = (prob >= 0.5).astype(int)

        rows.append({
            "condition": weight_condition,
            "fold": fold,
            "n_train": len(train),
            "n_valid": len(valid),
            "accuracy": accuracy_score(yva, pred),
            "precision": precision_score(yva, pred, zero_division=0),
            "recall": recall_score(yva, pred, zero_division=0),
            "specificity": specificity(yva, pred),
            "f1": f1_score(yva, pred, zero_division=0),
            "roc_auc": roc_auc_score(yva, prob),
            "pr_auc": average_precision_score(yva, prob),
            "balanced_accuracy": balanced_accuracy_score(yva, pred),
            "mcc": matthews_corrcoef(yva, pred),
        })

per_fold = pd.DataFrame(rows)
per_fold.to_csv(OUTDIR / "exp12_xgb_10fold_per_fold.csv", index=False)

summary = (
    per_fold
    .groupby("condition")
    .agg({
        "accuracy": ["mean","std"],
        "precision": ["mean","std"],
        "recall": ["mean","std"],
        "specificity": ["mean","std"],
        "f1": ["mean","std"],
        "roc_auc": ["mean","std"],
        "pr_auc": ["mean","std"],
        "balanced_accuracy": ["mean","std"],
        "mcc": ["mean","std"],
    })
)

display(summary)
summary.to_csv(OUTDIR / "exp12_xgb_10fold_summary.csv")


## 8. Interpretation

Primary direct comparison with Rao:
- use the `None` condition when the goal is to isolate **Bayesian KDE vs XGBoost**;
- report the `Balanced` condition separately as an adaptation of the thesis class-weighting strategy.

Do not compare these MGMT patient-level metrics directly with the thesis voxel Tumor/Non-tumor metrics as if they were the same task.
